In [ ]:
import os
import re
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
DATA_DIR = "csvs"
DB_PATH = "db/pipeline_db.sqlite"

In [ ]:
FILES = {
    "fact_table": "fact_table.csv",
    "customer_dim": "customer_dim.csv",
    "item_dim": "item_dim.csv",
    "store_dim": "store_dim.csv",
    "time_dim": "time_dim.csv",
    "trans_dim": "Trans_dim.csv",
}

FOREIGN_KEYS = {
    "customer_key": "customer_dim",
    "item_key": "item_dim",
    "store_key": "store_dim",
    "time_key": "time_dim",
    "payment_key": "trans_dim",
}

In [ ]:
def clean_name(name):
    name = re.sub(r"[^a-z0-9_]", "_", str(name).strip().lower())
    return re.sub(r"_+", "_", name).strip("_")


def read_csv(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")

In [ ]:
tables = {}
for table, filename in FILES.items():
    df = read_csv(os.path.join(DATA_DIR, filename))
    df.columns = [clean_name(c) for c in df.columns]
    tables[table] = df
    print(f"{table}: {len(df):,} rows, {len(df.columns)} columns")

fact = tables["fact_table"]
for key, dim in FOREIGN_KEYS.items():
    orphans = (~fact[key].isin(tables[dim][key])).sum()
    if orphans:
        raise ValueError(f"{orphans} rows in fact_table.{key} have no match in {dim}")
print("Foreign keys OK")

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
engine = create_engine(f"sqlite:///{DB_PATH}")
for table, df in tables.items():
    df.to_sql(table, engine, if_exists="replace", index=False)
engine.dispose()
print("Saved", DB_PATH)